## 🎯 Learning Objectives
* Understand the necessity and benefits of integrating external tools into AI agents.
* Learn how web search, calculator, and file reader tools enhance an agent's capabilities.
* Implement a basic AI agent capable of utilizing these three core tools using a modern framework.
* Interpret an agent's tool-use output and recognize common use cases and performance considerations.


## AG01-L06: Adding Tools: Web Search, Calculator, File Reader

Welcome to a pivotal lesson in building truly capable AI agents! So far, we've explored how Large Language Models (LLMs) form the 'brain' of our agents, enabling them to reason and generate text. However, even the most advanced LLMs have inherent limitations:

1.  **Knowledge Cut-off**: Their training data is static, meaning they lack real-time information about current events, stock prices, or recent scientific discoveries.
2.  **Factual Accuracy**: While LLMs are excellent at generating coherent text, they can 'hallucinate' facts or provide outdated information.
3.  **Complex Calculations**: LLMs are not designed for precise mathematical operations. Asking them to perform multi-step arithmetic or complex statistical analysis often leads to errors.
4.  **Access to External Data**: They cannot directly interact with local files, databases, or proprietary systems.

This is where **tools** come in. Imagine an LLM as a brilliant, highly articulate apprentice. This apprentice is incredibly smart, can learn quickly, and can explain complex concepts. But to perform real-world tasks, they need specialized equipment – tools. Just as a carpenter needs a saw, hammer, and measuring tape, an AI agent needs tools to extend its capabilities beyond pure language generation.

### The Power of Tools

Tools empower agents to:

*   **Perceive the Real World**: Access up-to-date information via web search.
*   **Perform Precise Actions**: Execute calculations, interact with APIs, or manipulate data.
*   **Access Private/Proprietary Data**: Read local documents, query databases, or interact with internal systems.

In this lesson, we'll focus on three fundamental and highly versatile tools:

1.  **Web Search**: The agent can query the internet to retrieve real-time information, verify facts, or gather context. This is like giving our apprentice access to the entire internet.
2.  **Calculator**: The agent can perform accurate mathematical operations, from simple arithmetic to complex expressions. This is like giving our apprentice a powerful, error-free calculator.
3.  **File Reader**: The agent can access and understand information stored in local files (e.g., `.txt`, `.pdf`, `.docx`). This is like giving our apprentice access to a company's internal reports or knowledge base.

### How Agents Use Tools: The `ReAct` Pattern

Modern agent frameworks often employ a `ReAct` (Reasoning and Acting) pattern. The agent observes a prompt, *reasons* about what steps are needed, *acts* by selecting and using a tool, *observes* the tool's output, and then *reasons* again based on the new information. This iterative loop allows agents to break down complex problems into smaller, manageable steps, leveraging tools as needed.

We'll use a popular framework like `LangChain` (or similar in 2026, which has evolved to be even more robust and integrated) to demonstrate how to equip an agent with these tools and observe its problem-solving process.


In [ ]:
# Ensure you have the necessary libraries installed. As of 2026, these are standard for agentic workflows.
# pip install langchain langchain-community langchain-core duckduckgo-search numexpr ollama

import os
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.llms import Ollama
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

# --- 1. Set up the LLM --- 
# We'll use Ollama for a local, privacy-preserving LLM. 
# Ensure you have Ollama installed and a model like 'llama3' pulled (ollama pull llama3).
# Alternatively, you can use a cloud-based LLM like Google Generative AI or OpenAI.
# For Google Generative AI:
# from langchain_google_genai import ChatGoogleGenerativeAI
# os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"
# llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=0)

llm = Ollama(model="llama3", temperature=0)

# --- 2. Define our Tools --- 

# Tool 1: Web Search (using DuckDuckGo for privacy and ease of use)
web_search_tool = DuckDuckGoSearchRun(name="WebSearch")

# Tool 2: Calculator (using numexpr for safe and efficient numerical evaluation)
@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression using numexpr. 
    Input should be a string representing a mathematical expression, e.g., '2 + 2 * 3'.
    """
    try:
        import numexpr
        return str(numexpr.evaluate(expression))
    except Exception as e:
        return f"Error evaluating expression: {e}"

# Tool 3: File Reader (a simple tool to read text files)
@tool
def file_reader(file_path: str) -> str:
    """Reads the content of a specified text file.
    Input should be the path to the file, e.g., 'report.txt'.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        return content
    except FileNotFoundError:
        return f"Error: File not found at {file_path}"
    except Exception as e:
        return f"Error reading file: {e}"

# Combine all tools into a list
tools = [web_search_tool, calculator, file_reader]

# --- 3. Create a Dummy File for the File Reader Tool --- 
# Let's create a sample report.txt file that our agent can read.
file_content = """
Project Alpha Q1 2026 Report

Key Findings:
- Revenue increased by 15% compared to Q4 2025.
- User acquisition grew by 20% due to successful marketing campaigns.
- Operating expenses were 1.2 million USD.
- Projected Q2 2026 revenue growth is estimated at 10%.
"""

with open("project_alpha_report.txt", "w", encoding="utf-8") as f:
    f.write(file_content)

print("Created 'project_alpha_report.txt' for the agent to read.\n")

# --- 4. Set up the Agent --- 

# Get the ReAct prompt from LangChain Hub. This prompt guides the LLM to use tools.
# As of 2026, prompt engineering for agents is highly standardized and often pulled from hubs.
prompt = hub.pull("hwchase17/react")

# Create the agent
agent = create_react_agent(llm, tools, prompt)

# Create the AgentExecutor to run the agent
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

# --- 5. Run the Agent with a Task --- 

# Define a complex task that requires all three tools
task = (
    "First, find the current market capitalization of 'NVIDIA' (NVDA). "
    "Then, read the 'project_alpha_report.txt' file to understand its key findings. "
    "Finally, calculate what 0.05% of NVIDIA's market capitalization would be, "
    "and combine this calculation with a summary of the project report's key findings."
)

print(f"Running agent with task:\n{task}\n")

# Invoke the agent
result = agent_executor.invoke({"input": task})

print("\n--- Agent's Final Answer ---")
print(result["output"])

# Clean up the dummy file
os.remove("project_alpha_report.txt")
print("\nCleaned up 'project_alpha_report.txt'.")


### Interpreting the Agent's Output and Use Cases

When you run the code above, pay close attention to the `verbose=True` output from the `AgentExecutor`. You'll see a detailed trace of the agent's thought process:

*   **Thought**: The agent's internal reasoning, explaining why it's choosing a particular tool or what its next step will be.
*   **Action**: The specific tool the agent decides to use (e.g., `WebSearch`, `calculator`, `file_reader`).
*   **Action Input**: The arguments passed to the chosen tool (e.g., a search query, a mathematical expression, a file path).
*   **Observation**: The output returned by the tool after execution. This is crucial as it provides new information for the agent to incorporate into its reasoning.

This `Thought-Action-Observation` loop demonstrates the agent's ability to dynamically plan and execute based on the task and available tools. The final `Agent's Final Answer` will be the synthesized response, incorporating information from all tool interactions.

### Performance Trade-offs and Use Cases

While incredibly powerful, integrating tools introduces several considerations:

*   **Latency**: External tool calls (especially web searches or API calls) add latency to the agent's response time. Each tool call is a round trip that takes time.
*   **Cost**: Cloud-based APIs for web search, specific data sources, or even LLM calls themselves can incur costs. Efficient tool usage is key to managing expenses.
*   **Reliability**: External services can be unavailable or return unexpected errors. Robust error handling within tools and the agent's reasoning is vital.
*   **Security**: When dealing with file systems or external APIs, security considerations (e.g., access control, input sanitization) become paramount.

**Typical Use Cases for these Tools (2026 Perspective):**

*   **Research & Reporting**: An agent can research a topic online, summarize findings from internal documents, and perform calculations for a comprehensive report.
*   **Financial Analysis**: Agents can fetch real-time stock data, read financial statements (PDFs), and perform complex financial modeling.
*   **Customer Support Automation**: Agents can search knowledge bases (internal files), look up product information online, and calculate refund amounts based on policies.
*   **Data Augmentation**: Agents can enrich datasets by fetching missing information from the web or internal data sources before processing.
*   **Content Generation**: An agent can research current trends, read source material, and then generate articles or summaries, ensuring factual accuracy and up-to-date information.

As we move further into 2026, agent frameworks are becoming even more sophisticated, offering advanced tool orchestration, multi-modal tool integration (e.g., image analysis tools), and self-healing capabilities for tool failures. The core principle, however, remains: tools are the agent's hands and eyes, extending its reach into the real world.


### Resources

*   **LangChain Tools Documentation**: [https://python.langchain.com/docs/modules/agents/tools/](https://python.langchain.com/docs/modules/agents/tools/)
*   **LangChain Agents Documentation**: [https://python.langchain.com/docs/modules/agents/](https://python.langchain.com/docs/modules/agents/)
*   **Ollama**: Run LLMs locally: [https://ollama.com/](https://ollama.com/)
*   **DuckDuckGo Search API (via LangChain)**: Integrated directly, but understanding the underlying service is useful: [https://duckduckgo.com/api](https://duckduckgo.com/api)
*   **NumExpr Documentation**: For fast numerical expression evaluation: [https://numexpr.readthedocs.io/en/latest/](https://numexpr.readthedocs.io/en/latest/)
*   **LangChain Hub**: Explore and share prompts and chains: [https://smith.langchain.com/hub](https://smith.langchain.com/hub)
